In [0]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor

df = spark.table("ifood_case.default.data_processing")

#-----------------------------------------
# Encode gender
#-----------------------------------------

# Manual gender encoding to avoid Spark Connect model size limit
df = df.withColumn(
    "gender_idx",
    F.when(F.col("gender") == "F", 0.0)
     .when(F.col("gender") == "M", 1.0)
     .when(F.col("gender") == "O", 2.0)
)

#-----------------------------------------
# Features
#-----------------------------------------

feature_cols = [
    "age",
    "gender_idx",
    "credit_card_limit",
    "registered_on_days",
    "total_transactions",
    "avg_ticket",
    "max_spent",
    "min_value",
    "duration",
    "discount_value",
    "has_web",
    "has_email",
    "has_mobile",
    "has_social"
]

df = df.fillna(0, subset=["has_web", "has_email", "has_mobile", "has_social", "discount_value", "duration", "min_value"])
df = df.fillna('no_offer', ['offer_type'])

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

df = assembler.transform(df)

In [0]:
from pyspark.sql.functions import abs
import mlflow
import os
import pyspark.sql.functions as F

prediction_df = df.select("account_id").drop_duplicates()
unique_client_df = df.drop_duplicates(['account_id'])

os.environ['MLFLOW_DFS_TMP'] = '/Volumes/ifood_case/default/mlflow_tmp'


for offer in ['informational', 'bogo', 'discount', 'no_offer']:
    if offer in ["no_offer", "discount"]:
        version = 5
    else: 
        version = 6
    test = spark.table(f'ifood_case.default.test_{offer}')

    model_uri = f'models:/ifood_case.default.rf_model_{offer}/{version}'
    pyspark_model = mlflow.spark.load_model(model_uri)

    # test = pyspark_model.transform(test)
    
    # # test = models[offer].transform(test)
    # test = test.withColumn("abs_error", abs(F.col("total_spent") - F.col("prediction")))

    # mae = test.agg(F.avg("abs_error").alias("MAE")).select("MAE").collect()[0]['MAE']
    # print(f'{offer} - MAE {mae}')

    pred = (
        pyspark_model.transform(unique_client_df)
        .select(
            "account_id",
            'total_spent',
            F.col("prediction").alias(
                f"pred_{offer}"
            )
        )
    )

    prediction_df = prediction_df.withColumnRenamed('total_spent',f'total_spent_{offer}')

    prediction_df = prediction_df.join(
        pred,
        on="account_id"
    )
    
    del pyspark_model
    del test

In [0]:
prediction_cols = [
    "pred_bogo",
    "pred_discount",
    "pred_informational",
    "pred_no_offer"
]

prediction_df = prediction_df.withColumn(
    "predictions",
    F.array(*[F.col(c) for c in prediction_cols])
)

prediction_df = prediction_df.withColumn(
    "max_prediction",
    F.array_max("predictions")
)

In [0]:
prediction_df = prediction_df.withColumn(
    "recommended_offer",
    F.when(
        F.col("max_prediction") == F.col("pred_bogo"),
        "bogo"
    )
    .when(
        F.col("max_prediction") == F.col("pred_discount"),
        "discount"
    )
    .when(
        F.col("max_prediction") == F.col("pred_informational"),
        "informational"
    )
    .otherwise("no_offer")
)

In [0]:
aux = spark.createDataFrame([], spark.table(f'ifood_case.default.test_{offer}').schema)
for offer in ['informational', 'bogo', 'discount', 'no_offer']:
    test = spark.table(f'ifood_case.default.test_{offer}')

    aux = aux.unionByName(test)

In [0]:
test_analise = aux.join(prediction_df.select('account_id', 'pred_bogo', 'pred_discount', 
                     'pred_no_offer', 'pred_informational', 'max_prediction', 'recommended_offer'), 
         on=['account_id'])

In [0]:
prediction_df_bogo = (test_analise
.withColumn("t", F.when(F.col("recommended_offer").isin( "bogo", "informational", "discount"), 1).otherwise(0)
).withColumn('uplift', F.col('max_prediction') - F.col('pred_no_offer')))

In [0]:
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
 
 
def qini_curve_spark(df: DataFrame, y_col="y", t_col="t",
                      score_col="uplift_score", n_bins: int = 100) -> DataFrame:
    """
    Retorna um Spark DataFrame pequeno (n_bins linhas) já pronto para
    collect()/toPandas() e plot, com colunas: frac, qini_real, qini_random.
    """
    n_total = df.count()
 
    # ordenar por score decrescente e gerar rank via window function
    w_order = Window.orderBy(F.col(score_col).desc())
 
    df_ranked = df.withColumn("rank", F.row_number().over(w_order))
 
    # acumulados: precisa de uma window "unbounded preceding until current row"
    # ordenada pelo mesmo critério
    w_cum = Window.orderBy("rank").rowsBetween(Window.unboundedPreceding, Window.currentRow)
 
    df_cum = (
        df_ranked
        .withColumn("y_treat", F.col(y_col) * F.col(t_col))
        .withColumn("y_ctrl", F.col(y_col) * (1 - F.col(t_col)))
        .withColumn("y_treat_cum", F.sum("y_treat").over(w_cum))
        .withColumn("y_ctrl_cum", F.sum("y_ctrl").over(w_cum))
        .withColumn("n_treat_cum", F.sum(F.col(t_col)).over(w_cum))
        .withColumn("n_ctrl_cum", F.sum(1 - F.col(t_col)).over(w_cum))
    )
 
    # totais para a linha de referência aleatória
    totals = df.agg(
        F.sum(t_col).alias("n_treat_total"),
        F.sum(1 - F.col(t_col)).alias("n_ctrl_total"),
    ).collect()[0]
    n_treat_total = totals["n_treat_total"] or 1
    n_ctrl_total = totals["n_ctrl_total"] or 1
 
    last_row = df_cum.orderBy(F.col("rank").desc()).select(
        "y_treat_cum", "y_ctrl_cum"
    ).limit(1).collect()[0]
    overall_uplift = (
        last_row["y_treat_cum"] / n_treat_total
        - last_row["y_ctrl_cum"] / n_ctrl_total
    )
 
    df_qini = (
        df_cum
        .withColumn("ratio", F.when(F.col("n_ctrl_cum") > 0,
                                     F.col("n_treat_cum") / F.col("n_ctrl_cum")).otherwise(0.0))
        .withColumn("qini_real", F.col("y_treat_cum") - F.col("y_ctrl_cum") * F.col("ratio"))
        .withColumn("frac", F.col("rank") / F.lit(n_total))
        .withColumn("qini_random", F.col("frac") * F.lit(n_treat_total) * F.lit(overall_uplift))
    )
 
    # reamostrar para n_bins pontos (evita coletar o dataset inteiro pro driver)
    bin_width = max(n_total // n_bins, 1)
    df_sampled = df_qini.filter((F.col("rank") % bin_width) == 0)
 
    return df_sampled.select("frac", "qini_real", "qini_random").orderBy("frac")
 
 
def auuc_score_spark(curve_df: DataFrame) -> float:
    """
    Area Under Uplift Curve via regra trapezoidal, calculada no driver
    a partir dos n_bins pontos já reamostrados (dataset pequeno).
    """
    import numpy as np
 
    pdf = curve_df.toPandas()
    trapezoid_fn = getattr(np, "trapezoid", None) or np.trapz
    return trapezoid_fn(pdf["qini_real"] - pdf["qini_random"], pdf["frac"])
 
 
def plot_qini_spark(curve_df: DataFrame, title="Qini Curve - Uplift do T-learner",
                     save_path=None):
    """
    Coleta os n_bins pontos para o driver e plota com matplotlib
    (mesma função de antes, só que a entrada agora vem de Spark).
    """
    import matplotlib.pyplot as plt
 
    pdf = curve_df.toPandas()
 
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(pdf["frac"], pdf["qini_real"], label="Modelo (T-learner)", linewidth=2)
    ax.plot(pdf["frac"], pdf["qini_random"], label="Envio aleatório (baseline)",
            linestyle="--", color="gray")
    ax.fill_between(pdf["frac"], pdf["qini_real"], pdf["qini_random"],
                     alpha=0.15, color="tab:blue")
    ax.set_xlabel("Fração da base ordenada por uplift estimado")
    ax.set_ylabel("Ganho incremental acumulado (conversões)")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
    return fig

In [0]:
curve = qini_curve_spark(prediction_df_bogo.drop_duplicates(['account_id']), 'total_spent', 't', 'uplift')
print("AUUC:", auuc_score_spark(curve))
plot_qini_spark(curve, save_path="qini_curve.png")